# 03 — Causal Inference Lab
De correlación a efecto causal sobre acciones comerciales.

Incluye diferencia cruda, regresión ajustada, propensity score, IPW, balance y gates causales.


In [1]:
from __future__ import annotations
import sys
from pathlib import Path
import pandas as pd
import numpy as np
cwd=Path.cwd().resolve()
PROJECT_ROOT=cwd if (cwd/"pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT/"pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")
SRC=PROJECT_ROOT/"src"
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))
from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres
settings=load_settings(PROJECT_ROOT)
conn=connect_postgres(settings)
def sql_df(sql,params=None):
    return pd.read_sql_query(sql,conn,params=params)
print("DB:",settings.postgres.database)

import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression


DB: medallio_dw


## 1. Dataset causal


In [2]:
causal=sql_df("""
SELECT
 r.recommendation_id,
 r.entity_id AS lead_id,
 r.created_at AS recommendation_at,
 a.action_taken,
 a.action_at,
 a.action_cost,
 le.codigo_proyecto,
 le.asesor,
 le.canal,
 le.medio,
 le.project_sep_rate_90d,
 le.advisor_sep_rate_90d,
 le.global_sep_rate_90d,
 le.separacion_14d,
 le.minuta_60d
FROM decision_intelligence.recommendations r
LEFT JOIN decision_intelligence.actions a
  ON a.recommendation_id=r.recommendation_id
LEFT JOIN features.lead_evidence le
  ON le.lead_id=r.entity_id
WHERE r.decision_system='priorizacion_leads'
""")
print("rows:",len(causal))
display(causal.head())


C:\Users\dinat\AppData\Local\Temp\ipykernel_36396\1232811218.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


DatabaseError: Execution failed on sql '
SELECT
 r.recommendation_id,
 r.entity_id AS lead_id,
 r.created_at AS recommendation_at,
 a.action_taken,
 a.action_at,
 a.action_cost,
 le.codigo_proyecto,
 le.asesor,
 le.canal,
 le.medio,
 le.project_sep_rate_90d,
 le.advisor_sep_rate_90d,
 le.global_sep_rate_90d,
 le.separacion_14d,
 le.minuta_60d
FROM decision_intelligence.recommendations r
LEFT JOIN decision_intelligence.actions a
  ON a.recommendation_id=r.recommendation_id
LEFT JOIN features.lead_evidence le
  ON le.lead_id=r.entity_id
WHERE r.decision_system='priorizacion_leads'
': no existe la columna r.created_at
LINE 5:  r.created_at AS recommendation_at,
         ^

## 2. Tratamiento y outcome


In [ ]:
if len(causal):
    causal["treatment"]=causal["action_taken"].notna().astype(int)
    causal["outcome"]=causal["minuta_60d"]
    display(causal[["treatment","outcome"]].describe())


## 3. Diferencia cruda


In [ ]:
if len(causal) and causal["outcome"].notna().any():
    crude=causal.groupby("treatment")["outcome"].agg(["count","mean"])
    display(crude)
    if set(crude.index)=={0,1}:
        print("ATE crudo:", crude.loc[1,"mean"]-crude.loc[0,"mean"])


## 4. Regresión ajustada


In [ ]:
reg=causal[["treatment","outcome","project_sep_rate_90d","advisor_sep_rate_90d","global_sep_rate_90d"]].dropna() if len(causal) else pd.DataFrame()
if len(reg)>30 and reg["treatment"].nunique()==2:
    model=smf.ols("outcome ~ treatment + project_sep_rate_90d + advisor_sep_rate_90d + global_sep_rate_90d",data=reg).fit(cov_type="HC3")
    print(model.summary())


## 5. Propensity Score + IPW


In [ ]:
ps_cols=["project_sep_rate_90d","advisor_sep_rate_90d","global_sep_rate_90d"]
ps_data=causal[["treatment","outcome"]+ps_cols].dropna() if len(causal) else pd.DataFrame()
if len(ps_data)>50 and ps_data["treatment"].nunique()==2:
    X=ps_data[ps_cols]
    t=ps_data["treatment"]
    ps_model=LogisticRegression(max_iter=1000).fit(X,t)
    ps_data["propensity"]=ps_model.predict_proba(X)[:,1]
    p=ps_data["propensity"].clip(.01,.99)
    y=ps_data["outcome"]
    w=t/p+(1-t)/(1-p)
    mu1=np.average(y[t==1],weights=w[t==1])
    mu0=np.average(y[t==0],weights=w[t==0])
    print("IPW ATE:",mu1-mu0)
    display(ps_data[["treatment","propensity"]].describe())


## 6. Balance de covariables


In [ ]:
if len(ps_data):
    rows=[]
    for c in ps_cols:
        g=ps_data.groupby("treatment")[c].agg(["mean","std"])
        if len(g)==2:
            pooled=np.sqrt((g.loc[0,"std"]**2+g.loc[1,"std"]**2)/2)
            smd=(g.loc[1,"mean"]-g.loc[0,"mean"])/pooled if pooled else np.nan
            rows.append({"covariate":c,"smd":smd})
    display(pd.DataFrame(rows))


## 7. Gate causal


In [ ]:
gates=[
{"gate":"Tratamiento/control disponibles","status":"PASS" if len(causal) and causal.get("treatment",pd.Series()).nunique()==2 else "PENDING"},
{"gate":"Outcome suficiente","status":"PASS" if len(causal) and causal.get("outcome",pd.Series()).notna().sum()>30 else "PENDING"},
{"gate":"Balance covariables","status":"PENDING"},
{"gate":"Ignorabilidad o asignación experimental defendible","status":"PENDING"},
]
pd.DataFrame(gates)


In [ ]:
conn.close(); print("Conexión cerrada.")
